In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Saved G receiver-origin compatibility diagnostic

Fixed G run 20261005T140121854219Z: saved FULL181 MP4-readback RGB P0/P1, each cut at [0:177] and [1:178]. No new video, writer, framewise VAE, codec, sync search, or offset restoration. Exactly four Wan receiver encodes, eight K0/K1 reads, R44, 1320 votes/bit, 337920 detailed votes, 11264 time-bit rows, 256 final bits. Missing input or identity mismatch keeps failed slots; there is no codec fallback or retry selection.

The original reader is unchanged. Signed votes mean 2*int(FFT.real>0)-1; exact coefficient zero gives -1, not erasure. Original time-major/frequency-subsequence order and Counter first-encounter tie decisions are retained. Receiver latent indices and nominal stride4 coordinates are indexing labels, not independent four-frame supports or definite source-frame attribution. Current readouts are sealed before joining condition/start/truth and before reading historical payload. Historical differences do not trigger reruns. This is a bounded compatibility diagnostic, not proof of phase causality, an offset-recovery implementation, FPR, or scientific PASS.

The notebook has only CPU/fake/static validation before handoff. Real Run all is for the user after publication. Relevant dependency versions follow the saved G environment; pip check warnings remain recorded, with no unrelated repairs.

Fixed inputs:
- P0_ORIGINAL_RGB: /content/drive/MyDrive/Video-WM/Trajectory-Payload-G-V1/20261005T140121854219Z/fixed_reference/P0_ORIGINAL_RGB/received.rgb8; SHA256 127aec31b5f595e046e05e1190dc77a571a01dfa333522da5421c1049be23e9d
- P1_FRAMEWISE_RECON: /content/drive/MyDrive/Video-WM/Trajectory-Payload-G-V1/20261005T140121854219Z/fixed_reference/P1_FRAMEWISE_RECON/received.rgb8; SHA256 a7e062fffb8fef46d92c2559a5bda49b4e5c7a822f1630b94eb8e2dc773e61dd

UNPUBLISHED DRAFT: SOURCE_SHA=None; publish reviewed source and bind its immutable SHA before Run all.

In [ ]:
SOURCE_SHA = None
FIXED = {'observations': 4, 'wan_receiver_encode': 4, 'payload_reads': 8, 'keys': 2, 'bits_per_read': 32, 'R': 44, 'frequency_positions_per_bit': 30, 'votes_per_bit': 1320, 'votes_per_key_read': 42240, 'detailed_votes': 337920, 'time_bit_rows': 11264, 'final_bit_rows': 256}
from pathlib import Path
import datetime,hashlib,json,os,signal,subprocess,sys,time,traceback
if SOURCE_SHA is None:
    raise RuntimeError('UNPUBLISHED_DRAFT: publish reviewed source and rebuild with its immutable SHA before Run all')
if not isinstance(SOURCE_SHA,str) or len(SOURCE_SHA)!=40:
    raise RuntimeError('immutable 40-character source SHA required')
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=Path('/content/drive/MyDrive/Video-WM/Trajectory-Receiver-Origin-V1')/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
RUN_OUTPUT=OUTPUT/'fixed_reference'
REPO=Path('/content/SC-SSTW-TRAJECTORY-RECEIVER-ORIGIN-V1-'+STAMP)
PYTHON=sys.executable
def write_json(path,value):
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(value,indent=2)+'\n');os.replace(tmp,path)
def failed(stage,exc):
    captured_traceback=''.join(traceback.format_exception(type(exc),exc,exc.__traceback__))
    write_json(OUTPUT/'setup_failure.json',dict(stage=stage,error=f'{type(exc).__name__}: {exc}',traceback=captured_traceback,fixed_denominator=FIXED))
def logged(command,stage,cwd=None,check=True):
    child=None;code=None;primary=None;primary_tb=None;cleanup_errors=[]
    def retain_cleanup_error(label,exc):
        nonlocal primary,primary_tb
        if primary is None and not isinstance(exc,Exception):
            primary=exc;primary_tb=exc.__traceback__
        else:
            cleanup_errors.append(label+': '+repr(exc))
    try:
        with (OUTPUT/'execution.log').open('a') as log:
            log.write('COMMAND '+repr(command)+'\n');log.flush()
            popen_group={'start_new_session':True} if os.name=='posix' else {}
            child=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,**popen_group)
            try:
                for line in child.stdout:
                    print(line,end='',flush=True);log.write(line);log.flush()
                code=child.wait()
            except BaseException as exc:
                primary=exc;primary_tb=exc.__traceback__
            finally:
                if child is not None:
                    if os.name=='posix':
                        pgid=child.pid
                        try:os.killpg(pgid,signal.SIGTERM)
                        except ProcessLookupError:pass
                        except BaseException as exc:retain_cleanup_error('group SIGTERM',exc)
                        try:
                            deadline=time.monotonic()+5.0
                            while time.monotonic()<deadline:
                                try:os.killpg(pgid,0)
                                except ProcessLookupError:break
                                except BaseException as exc:
                                    retain_cleanup_error('group probe',exc);break
                                time.sleep(0.05)
                        except BaseException as exc:
                            retain_cleanup_error('group TERM grace',exc)
                        finally:
                            try:os.killpg(pgid,signal.SIGKILL)
                            except ProcessLookupError:pass
                            except BaseException as exc:retain_cleanup_error('group SIGKILL',exc)
                    elif child.poll() is None:
                        try:child.terminate()
                        except ProcessLookupError:pass
                        except BaseException as exc:retain_cleanup_error('terminate',exc)
                    try:
                        try:code=child.wait(timeout=10)
                        except subprocess.TimeoutExpired:
                            try:child.kill()
                            except ProcessLookupError:pass
                            except BaseException as exc:retain_cleanup_error('kill',exc)
                            try:code=child.wait(timeout=10)
                            except BaseException as exc:retain_cleanup_error('wait after kill',exc)
                        except BaseException as exc:retain_cleanup_error('wait',exc)
                    finally:
                        if child.stdout is not None:
                            try:child.stdout.close()
                            except BaseException as exc:retain_cleanup_error('stdout close',exc)
            if primary is None and not cleanup_errors:
                log.write('EXIT '+str(code)+'\n');log.flush()
    except BaseException as wrapper_error:
        if primary is None:
            primary=wrapper_error;primary_tb=wrapper_error.__traceback__
        elif wrapper_error is not primary:
            cleanup_errors.append('log wrapper/close: '+repr(wrapper_error))
    if primary is None and cleanup_errors:
        primary=RuntimeError('process cleanup failed: '+'; '.join(cleanup_errors))
        primary_tb=primary.__traceback__
    if primary is None and check and code:
        primary=subprocess.CalledProcessError(code,command)
        primary_tb=primary.__traceback__
    if primary is not None:
        if cleanup_errors and hasattr(primary,'add_note'):
            primary.add_note('secondary cleanup errors: '+'; '.join(cleanup_errors))
        try:failed(stage,primary)
        except BaseException as record_error:
            if hasattr(primary,'add_note'):primary.add_note('failure record error: '+repr(record_error))
        raise primary.with_traceback(primary_tb)
    return code
write_json(OUTPUT/'setup_receipt.json',dict(status='SETUP_STARTED',source_sha=SOURCE_SHA,fixed_denominator=FIXED))


In [ ]:
try:
    logged(['git','clone','--filter=blob:none','https://github.com/RICHAAARC/SC-SSTW.git',str(REPO)],'SOURCE_CLONE')
    logged(['git','-C',str(REPO),'fetch','origin',SOURCE_SHA],'SOURCE_FETCH')
    logged(['git','-C',str(REPO),'checkout','--detach',SOURCE_SHA],'SOURCE_CHECKOUT')
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
    if actual!=SOURCE_SHA or dirty:raise RuntimeError('source SHA/clean checkout mismatch')
    probe="import importlib.metadata as m; pins={'torch': '2.11.0', 'diffusers': '0.39.0', 'transformers': '4.57.6', 'numpy': '2.1.3', 'accelerate': '1.15.0', 'safetensors': '0.8.0', 'huggingface-hub': '0.36.2', 'tokenizers': '0.22.2'}; actual={k:m.version(k) for k in pins}; assert all(actual[k].split('+')[0]==v for k,v in pins.items()), actual; from diffusers import AutoencoderKLWan; import torch; print(actual)"
    if logged([PYTHON,'-c',probe],'RECEIVER_DEPENDENCY_PROBE',check=False):
        logged([PYTHON,'-m','pip','install','torch==2.11.0','diffusers==0.39.0','transformers==4.57.6','numpy==2.1.3','accelerate==1.15.0','safetensors==0.8.0','huggingface-hub==0.36.2','tokenizers==0.22.2'],'RECEIVER_DEPENDENCY_REPAIR')
        logged([PYTHON,'-c',probe],'RECEIVER_DEPENDENCY_REPROBE')
    dependency_code=logged([PYTHON,'-m','pip','check'],'DEPENDENCY_REPORT',check=False)
    (OUTPUT/'environment_freeze.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
    write_json(OUTPUT/'source_receipt.json',dict(source_sha=actual,clean=True))
    write_json(OUTPUT/'setup_receipt.json',dict(status='SETUP_COMPLETE',source_sha=actual,fixed_denominator=FIXED,dependency_check_returncode=dependency_code))
except Exception as exc:
    try:failed('SOURCE_OR_ENVIRONMENT',exc)
    except BaseException as record_error:
        if hasattr(exc,'add_note'):exc.add_note('failure record error: '+repr(record_error))
    raise


In [ ]:
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
    if token:os.environ['HF_TOKEN']=token
except Exception:pass
finally:token=None
command=[PYTHON,'-u','-m','experiments.wan_state_clock.video_trajectory_receiver_origin_v1_run','--output',str(RUN_OUTPUT)]
try:
    returncode=logged(command,'FIXED_RUN',cwd=REPO,check=False)
    RESULT_PATH=RUN_OUTPUT/'result.json'
    write_json(OUTPUT/'execution_receipt.json',dict(command=command,returncode=returncode,result_path=str(RESULT_PATH),result_exists=RESULT_PATH.is_file()))
    if not RESULT_PATH.is_file():raise RuntimeError('No runner result; fixed denominator retained')
except Exception as exc:
    try:failed('RUNNER',exc)
    except BaseException as record_error:
        if hasattr(exc,'add_note'):exc.add_note('failure record error: '+repr(record_error))
    raise


In [ ]:
import statistics
result=json.loads(RESULT_PATH.read_text())
if result['source_sha']!=SOURCE_SHA or result['fixed_denominator']!=FIXED:raise RuntimeError('result identity mismatch')
print('Status:',result['status'],'Counts:',result['counts'],'Calls:',result['calls'])
print('Actual environment:',result['environment'])
print('Blind seal:',result.get('blind_receiver_sha256'))
for sid,row in sorted(result['payload_posthoc'].items()):
    bits=row.get('bit_rows',[])
    valid=[x for x in bits if 'signed_normalized_margin' in x]
    print(sid,row['status'],'condition=',row.get('condition'),'start=',row.get('source_start_posthoc_only'),
          'key=',row.get('key_role'),'errors=',row.get('error_bits'),
          'signed margin min=',min((x['signed_normalized_margin'] for x in valid),default=None))
    print('  per-payload-channel medians:',[
        statistics.median([x['signed_normalized_margin'] for x in valid if x['payload_channel']==ch])
        if any(x['payload_channel']==ch for x in valid) else None for ch in range(4)])
    print('  complete vote/time artifacts:',result['payload_reads'][sid]['detail_path'],row.get('time_bit_path'))
for sid,row in sorted(result['historical_comparisons'].items()):
    print('Historical start1, postseal only:',sid,row['status'],'decoded_equal=',row.get('decoded_equal'),'votes_equal=',row.get('votes_equal'))
print('Failures:',result['failures'])
print('Historical reference error:',result.get('historical_reference_error'))
print('Evidence ceiling:',result['evidence_ceiling'])
print('Result:',RESULT_PATH)
